# ANA500 Micro-Project 1: Adult Income Dataset
### Acquire and Prepare using NumPy and Pandas

**Course:** ANA500  
**Micro-Project:** 1 of 4  
**Dataset:** Adult / Census Income (UCI Machine Learning Repository, ID 2)

---

### Scope of this notebook

Per the assignment specification, Micro-Project 1 terminates at **Step 2 (Prepare)** of the
data science process. This notebook therefore covers:

1. Problem statement and hypothesis (stated, not tested)
2. **Acquire** - source identification, retrieval, provenance, structural inspection
3. **Prepare** - exploration, quality auditing, cleaning, and feature construction

Modeling, evaluation, and interpretation are deliberately out of scope and are carried
forward to Micro-Projects 2 through 4.

---

### Problem statement

Earnings in the 1994 United States workforce were distributed unevenly across demographic
and employment characteristics. Individuals with comparable working hours reached very
different income levels, and the attributes associated with crossing the 50,000 dollar
annual earnings threshold are not directly observable from any single variable. The
phenomenon this project describes is which recorded personal and employment attributes
are associated with an individual earning above that threshold.

### Hypothesis

Educational attainment and weekly hours worked are the dominant correlates of exceeding
50,000 dollars in annual income, with occupation category and marital status contributing
additional explanatory signal beyond those two variables.

This hypothesis is stated here to direct the preparation work. It is **not** tested in this
notebook, since the project scope ends at Prepare. The deliverable is an analysis-ready
dataset capable of supporting that test in a later micro-project.

## 0. Environment setup

Only NumPy and Pandas are used for the analysis itself, as required by the Micro-Project 1
specification. No modeling or visualization libraries are imported.

In [1]:
# ---------------------------------------------------------------------------
# Core libraries. The Micro-Project 1 specification restricts this project to
# NumPy and Pandas objects for organizing, cleaning, and analyzing the data.
# ---------------------------------------------------------------------------
import io
import socket
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

# Fail fast rather than hanging if a remote host is unreachable.
socket.setdefaulttimeout(20)

# Display options: show every column and avoid scientific notation, so that
# quality checks below are legible in the knitted output.
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

# Fixed seed. No random operations are used in this notebook, but the seed is
# set so that any sampling added later is reproducible.
RNG = np.random.default_rng(500)

print("pandas:", pd.__version__)
print("numpy :", np.__version__)

pandas: 3.0.2
numpy : 2.4.4


---

# Step 1. ACQUIRE

## 1.1 Source and provenance

| Item | Detail |
|---|---|
| Dataset | Adult, also published as Census Income |
| Repository | UCI Machine Learning Repository, dataset ID 2 |
| Citation | Becker, B. and Kohavi, R. (1996). *Adult* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5XW20 |
| License | Creative Commons Attribution 4.0 International (CC BY 4.0) |
| Origin | Extracted by Barry Becker from the 1994 United States Census database |
| Extraction filter | age > 16, adjusted gross income > 100, final weight > 1, hours worked > 0 |
| Instances | 48,842 |
| Features | 14 predictors plus 1 target |
| Prediction task | Binary classification of income above or below 50,000 dollars per year |

The repository distributes the data as two files, `adult.data` (32,561 rows) and
`adult.test` (16,281 rows), neither of which carries a header row. The two files are a
train and test split of a single extract rather than two different populations, so they are
concatenated here and the split is deferred to the modeling stage in a later micro-project.

Three retrieval quirks in the raw files must be handled at load time:

1. **No header row.** Column names are supplied manually from the repository documentation.
2. **Missing values are encoded as the literal string `?`,** not as an empty field.
3. **`adult.test` carries a non-data first line and appends a period to every target label,**
   which produces four distinct target values instead of two if the files are merged naively.

In [2]:
# ---------------------------------------------------------------------------
# Column names. The raw repository files have no header row, so the schema is
# taken from the dataset documentation. Names are converted to snake_case here
# so that they are valid Python identifiers downstream.
# ---------------------------------------------------------------------------
COLUMNS = [
    "age", "workclass", "fnlwgt", "education", "education_num",
    "marital_status", "occupation", "relationship", "race", "sex",
    "capital_gain", "capital_loss", "hours_per_week", "native_country", "income",
]

# Canonical repository locations, tried first.
UCI_TRAIN = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
UCI_TEST  = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test"

# Mirror of the same extract, already concatenated, used only if the repository
# is unreachable from the execution environment.
MIRROR = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/adult-all.csv"

# Local cache path, checked first so that the notebook is re-runnable offline.
LOCAL = Path("data/adult-all.csv")


def _read_raw(source, skiprows=0):
    '''Read one raw Adult file into a DataFrame.

    skipinitialspace strips the leading blank that the repository files place
    after every comma. na_values maps the sentinel to a true NaN so that
    Pandas missing-value tooling works on it.
    '''
    return pd.read_csv(
        source,
        header=None,             # raw files carry no header row
        names=COLUMNS,           # schema supplied from documentation
        skiprows=skiprows,       # adult.test has one non-data leading line
        skipinitialspace=True,   # remove the leading space on every field
        na_values=["?"],         # convert the sentinel to NaN at parse time
    )


def load_adult():
    '''Return the full 48,842-row extract plus a note on which source was used.'''
    # Preferred path: the two canonical repository files, concatenated.
    try:
        train = _read_raw(UCI_TRAIN)
        test = _read_raw(UCI_TEST, skiprows=1)   # skip the '|1x3 Cross validator' line
        raw = pd.concat([train, test], ignore_index=True)
        return raw, f"UCI repository ({len(train):,} train + {len(test):,} test rows)"
    except Exception as exc:
        print(f"UCI repository unavailable ({type(exc).__name__}); falling back.")

    # Fallback 1: local cache, if a previous run already saved one.
    if LOCAL.exists():
        return _read_raw(LOCAL), f"local cache at {LOCAL}"

    # Fallback 2: GitHub mirror of the identical extract.
    raw = _read_raw(MIRROR)
    LOCAL.parent.mkdir(parents=True, exist_ok=True)
    raw.to_csv(LOCAL, header=False, index=False)   # cache for re-runs
    return raw, "GitHub mirror of the UCI extract"


df_raw, SOURCE_USED = load_adult()
print(f"Loaded from: {SOURCE_USED}")
print(f"Shape      : {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

UCI repository unavailable (HTTPError); falling back.


Loaded from: local cache at data/adult-all.csv
Shape      : 48,842 rows x 15 columns


## 1.2 Structural inspection

Before any cleaning decision is made, the raw object is inspected for shape, storage types,
and memory footprint. This establishes the baseline that the Prepare step is measured
against.

In [3]:
# ---------------------------------------------------------------------------
# First five records, as loaded. This is a visual confirmation that the column
# names align with the values and that the parser handled the leading spaces.
# ---------------------------------------------------------------------------
df_raw.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


In [4]:
# ---------------------------------------------------------------------------
# Storage types and non-null counts. df.info() is the fastest single view of
# whether Pandas inferred the intended type for each column and where missing
# values are concentrated.
# ---------------------------------------------------------------------------
df_raw.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 48842 entries, 0 to 48841
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   age             48842 non-null  int64
 1   workclass       46043 non-null  str  
 2   fnlwgt          48842 non-null  int64
 3   education       48842 non-null  str  
 4   education_num   48842 non-null  int64
 5   marital_status  48842 non-null  str  
 6   occupation      46033 non-null  str  
 7   relationship    48842 non-null  str  
 8   race            48842 non-null  str  
 9   sex             48842 non-null  str  
 10  capital_gain    48842 non-null  int64
 11  capital_loss    48842 non-null  int64
 12  hours_per_week  48842 non-null  int64
 13  native_country  47985 non-null  str  
 14  income          48842 non-null  str  
dtypes: int64(6), str(9)
memory usage: 26.4 MB


In [5]:
# ---------------------------------------------------------------------------
# Baseline metrics, captured now so that the effect of every cleaning step can
# be quantified at the end of the Prepare stage.
# ---------------------------------------------------------------------------
BASELINE = {
    "rows": len(df_raw),
    "cols": df_raw.shape[1],
    "memory_mb": df_raw.memory_usage(deep=True).sum() / 1024**2,
    "missing_cells": int(df_raw.isna().sum().sum()),
    "duplicate_rows": int(df_raw.duplicated().sum()),
}

for key, value in BASELINE.items():
    print(f"{key:>16}: {value:,.2f}" if isinstance(value, float) else f"{key:>16}: {value:,}")

            rows: 48,842
            cols: 15
       memory_mb: 26.36
   missing_cells: 6,465
  duplicate_rows: 52


## 1.3 Data dictionary

| Column | Type | Description |
|---|---|---|
| `age` | integer | Age in years, filtered to over 16 at extraction |
| `workclass` | categorical | Employer type, for example Private or Federal-gov |
| `fnlwgt` | integer | **Final weight.** Census sampling weight, not a personal attribute |
| `education` | categorical | Highest level attained, as a text label |
| `education_num` | integer | Highest level attained, as an ordinal code |
| `marital_status` | categorical | Marital status at time of survey |
| `occupation` | categorical | Occupational category |
| `relationship` | categorical | Role within the household |
| `race` | categorical | Self-reported race |
| `sex` | categorical | Self-reported sex as recorded in 1994 |
| `capital_gain` | integer | Capital gains in dollars, top-coded at 99999 |
| `capital_loss` | integer | Capital losses in dollars |
| `hours_per_week` | integer | Usual hours worked per week |
| `native_country` | categorical | Country of origin |
| `income` | binary target | Whether annual income exceeds 50,000 dollars |

Two entries in this dictionary drive cleaning decisions taken later in the Prepare step.
`fnlwgt` is a survey design weight describing how many people in the population a record
represents, so it is not a characteristic of the individual. `education` and
`education_num` describe the same underlying variable in two encodings.

---

# Step 2. PREPARE

The Prepare step is organized as an audit followed by a set of remediations. Every
remediation is stated with the evidence that motivated it, because an unjustified cleaning
decision silently changes the population under study.

## 2.1 Structural normalization

Three normalizations are applied before any quality metric is computed, so that the metrics
are not distorted by formatting artifacts.

In [6]:
# ---------------------------------------------------------------------------
# Work on a copy. The raw object is retained unmodified so that before-and-after
# comparisons at the end of this notebook are valid.
# ---------------------------------------------------------------------------
df = df_raw.copy()

# --- Normalization 1: strip residual whitespace from every text column -------
# skipinitialspace handled the leading space at parse time, but trailing
# whitespace can survive it. Selecting on both 'object' and 'string' keeps this
# correct across Pandas versions, since Pandas 3 stores text as a str dtype
# rather than object by default.
text_cols = df.select_dtypes(include=["object", "string"]).columns
for col in text_cols:
    df[col] = df[col].str.strip()

# --- Normalization 2: re-apply the '?' sentinel ------------------------------
# na_values caught the sentinel at parse time, but a value such as ' ?' with an
# unusual spacing pattern could survive. np.where re-checks every text cell and
# replaces any remaining sentinel with a true missing value.
for col in text_cols:
    df[col] = pd.Series(
        np.where(df[col].isin(["?", ""]), np.nan, df[col]),
        index=df.index,
    )

# --- Normalization 3: harmonize the target -----------------------------------
# adult.test appends a period to every label, so a naive concatenation produces
# four target values ('<=50K', '>50K', '<=50K.', '>50K.') instead of two.
df["income"] = df["income"].str.rstrip(".").str.strip()

print("Distinct target values after normalization:", sorted(df["income"].dropna().unique()))
print("\nTarget distribution:")
print(df["income"].value_counts())
print("\nAs a percentage:")
print((df["income"].value_counts(normalize=True) * 100).round(2))

Distinct target values after normalization: ['<=50K', '>50K']

Target distribution:
income
<=50K    37155
>50K     11687
Name: count, dtype: int64

As a percentage:
income
<=50K   76.07
>50K    23.93
Name: proportion, dtype: float64


The target is imbalanced at roughly 76 percent to 24 percent. This is recorded here as a
property of the data and is **not** corrected in this notebook. Resampling or class
weighting is a modeling decision, and applying it during preparation would contaminate any
later evaluation split.

## 2.2 Duplicate records

The extract is a survey sample, so two respondents can legitimately share every recorded
attribute without being the same person. Exact duplicates are nevertheless removed, because
a repeated row is indistinguishable from a data entry artifact and gives one observation
double weight in any later fit.

In [7]:
# ---------------------------------------------------------------------------
# Count exact full-row duplicates, then inspect one before removing anything.
# ---------------------------------------------------------------------------
n_dupes = int(df.duplicated().sum())
print(f"Exact duplicate rows: {n_dupes:,} ({n_dupes / len(df) * 100:.3f}% of the extract)")

# Show a duplicated pair so that the removal is auditable rather than blind.
if n_dupes:
    example = df[df.duplicated(keep=False)].sort_values(list(df.columns)).head(2)
    print("\nExample of a duplicated pair:")
    display(example)

# Remove duplicates, keeping the first occurrence of each.
rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
POST_DEDUP_ROWS = len(df)   # retained for the validation step in section 2.11
print(f"\nRows: {rows_before:,} -> {len(df):,} ({rows_before - len(df):,} removed)")

Exact duplicate rows: 52 (0.106% of the extract)

Example of a duplicated pair:


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
24667,17,Private,153021,12th,8,Never-married,Sales,Own-child,White,Female,0,0,20,United-States,<=50K
36713,17,Private,153021,12th,8,Never-married,Sales,Own-child,White,Female,0,0,20,United-States,<=50K



Rows: 48,842 -> 48,790 (52 removed)


## 2.3 Missing value audit

Missing values in this extract are confined to three columns. The remediation depends
entirely on **why** they are missing, so that question is answered with evidence before a
strategy is chosen.

In [8]:
# ---------------------------------------------------------------------------
# Per-column missingness, expressed in both absolute and relative terms.
# ---------------------------------------------------------------------------
missing_n = df.isna().sum()
missing_report = pd.DataFrame({
    "n_missing": missing_n,
    "pct_missing": (missing_n / len(df) * 100).round(2),
})
missing_report = missing_report[missing_report["n_missing"] > 0]
print("Columns containing missing values:")
display(missing_report)

# Row-level view: how much of the extract would listwise deletion discard?
rows_any_na = df.isna().any(axis=1)
print(f"Rows with at least one missing value: {rows_any_na.sum():,} "
      f"({rows_any_na.mean() * 100:.2f}% of the extract)")

Columns containing missing values:


,n_missing,pct_missing
workclass,2795,5.73
occupation,2805,5.75
native_country,856,1.75


Rows with at least one missing value: 3,615 (7.41% of the extract)


In [9]:
# ---------------------------------------------------------------------------
# Test 1: is the missingness structural? If the missing cells co-occur in a
# fixed pattern rather than scattering at random, they encode a real state
# rather than a recording failure.
# ---------------------------------------------------------------------------
na_workclass = df["workclass"].isna()
na_occupation = df["occupation"].isna()

print(f"workclass missing            : {na_workclass.sum():,}")
print(f"occupation missing           : {na_occupation.sum():,}")
print(f"both missing on the same row : {(na_workclass & na_occupation).sum():,}")
print(f"occupation missing only      : {(na_occupation & ~na_workclass).sum():,}")
print(f"workclass missing only       : {(na_workclass & ~na_occupation).sum():,}")

# Inspect the workclass values on the rows where only occupation is missing.
print("\nworkclass values where occupation alone is missing:")
print(df.loc[na_occupation & ~na_workclass, "workclass"].value_counts())

workclass missing            : 2,795
occupation missing           : 2,805
both missing on the same row : 2,795
occupation missing only      : 10
workclass missing only       : 0

workclass values where occupation alone is missing:
workclass
Never-worked    10
Name: count, dtype: int64


**Finding.** The missingness is structural, not random. Every row missing `workclass` is
also missing `occupation`, and the only rows missing `occupation` alone are exactly those
whose `workclass` is `Never-worked` - a person who has never worked cannot have an
occupation. The missing cells therefore encode a genuine employment state.

In [10]:
# ---------------------------------------------------------------------------
# Test 2: is the missingness informative with respect to the target? If the
# target rate differs between complete and incomplete rows, then discarding the
# incomplete rows would shift the very quantity the project sets out to explain.
# ---------------------------------------------------------------------------
target_positive = df["income"].eq(">50K")

rate_incomplete = target_positive[rows_any_na].mean() * 100
rate_complete = target_positive[~rows_any_na].mean() * 100

print(f">50K rate, rows with missing values : {rate_incomplete:.2f}%")
print(f">50K rate, complete rows            : {rate_complete:.2f}%")
print(f"Absolute difference                 : {abs(rate_complete - rate_incomplete):.2f} "
      f"percentage points")

>50K rate, rows with missing values : 13.25%
>50K rate, complete rows            : 24.80%
Absolute difference                 : 11.55 percentage points


**Finding.** The positive-class rate among incomplete rows is roughly half that of complete
rows. Listwise deletion would therefore discard a subgroup that is systematically poorer
than the retained population, biasing the target distribution upward.

**Decision.** Missing categorical values are recoded to an explicit `Unknown` level rather
than dropped or imputed. Dropping would bias the sample as shown above. Mode imputation
would be worse still, since it would assert a specific employer type and occupation for
people who are structurally outside the labor force. An explicit level preserves every row
and lets a later model use the missingness itself as signal.

In [11]:
# ---------------------------------------------------------------------------
# Apply the decision: fill the three affected categorical columns with an
# explicit 'Unknown' level. np.where makes the substitution visible rather than
# hiding it inside fillna, and keeps the operation identical across columns.
# ---------------------------------------------------------------------------
UNKNOWN = "Unknown"
cols_with_na = ["workclass", "occupation", "native_country"]

for col in cols_with_na:
    df[col] = pd.Series(
        np.where(df[col].isna(), UNKNOWN, df[col]),
        index=df.index,
    )

print(f"Missing cells remaining in the frame: {int(df.isna().sum().sum())}")
print("\nworkclass levels after recoding:")
print(df["workclass"].value_counts())

Missing cells remaining in the frame: 0

workclass levels after recoding:
workclass
Private             33860
Self-emp-not-inc     3861
Local-gov            3136
Unknown              2795
State-gov            1981
Self-emp-inc         1694
Federal-gov          1432
Without-pay            21
Never-worked           10
Name: count, dtype: int64


## 2.4 Redundant encoding: `education` and `education_num`

The data dictionary suggests that these two columns describe the same variable. That
suspicion is verified rather than assumed, because dropping a column on a guess would
discard information.

In [12]:
# ---------------------------------------------------------------------------
# A bijection test. If each ordinal code maps to exactly one text label and each
# text label maps to exactly one ordinal code, the two columns carry identical
# information and one of them is redundant.
# ---------------------------------------------------------------------------
labels_per_code = df.groupby("education_num")["education"].nunique()
codes_per_label = df.groupby("education")["education_num"].nunique()

print(f"Maximum distinct labels for any one ordinal code: {labels_per_code.max()}")
print(f"Maximum distinct ordinal codes for any one label: {codes_per_label.max()}")

is_bijection = (labels_per_code.max() == 1) and (codes_per_label.max() == 1)
print(f"\nOne-to-one mapping confirmed: {is_bijection}")

# Print the full mapping so the ordinal ordering can be inspected for sanity.
mapping = (df[["education_num", "education"]]
           .drop_duplicates()
           .sort_values("education_num")
           .reset_index(drop=True))
print("\nFull ordinal mapping:")
display(mapping)

Maximum distinct labels for any one ordinal code: 1
Maximum distinct ordinal codes for any one label: 1

One-to-one mapping confirmed: True

Full ordinal mapping:


,education_num,education
0,1,Preschool
1,2,1st-4th
2,3,5th-6th
3,4,7th-8th
4,5,9th
5,6,10th
6,7,11th
7,8,12th
8,9,HS-grad
9,10,Some-college


**Decision.** The mapping is one to one and the ordinal codes are correctly ordered from
`Preschool` at 1 to `Doctorate` at 16. `education_num` is retained because it is already a
model-ready ordinal encoding that preserves the natural ordering of attainment.
`education` is dropped. The text labels are preserved in the mapping table above so that
results can be reported in human-readable terms later.

In [13]:
# ---------------------------------------------------------------------------
# Retain the mapping as a lookup object before dropping the text column, so the
# labels remain available for reporting in later micro-projects.
# ---------------------------------------------------------------------------
EDUCATION_LOOKUP = dict(zip(mapping["education_num"], mapping["education"]))

df = df.drop(columns=["education"])
print(f"Dropped 'education'. Columns remaining: {df.shape[1]}")

Dropped 'education'. Columns remaining: 14


## 2.5 `fnlwgt`: a survey weight, not a personal attribute

`fnlwgt` is the Census Bureau final weight. It estimates how many people in the wider
population each sampled record represents, derived from the sampling design and
post-stratification controls. It is a property of the *survey*, not of the *person*.

Including it as a predictor is a common error with this dataset. It is not a leak in the
usual sense, but it is meaningless as a personal characteristic, and any apparent
relationship it shows with income is an artifact of the sampling frame rather than a fact
about individuals.

In [14]:
# ---------------------------------------------------------------------------
# Evidence check: if fnlwgt described the individual, it would show some
# correlation with other personal attributes. Near-zero correlation across all
# of them is consistent with it being a design weight.
# ---------------------------------------------------------------------------
print("fnlwgt summary statistics:")
display(df["fnlwgt"].describe())

correlations = (df[["fnlwgt", "age", "education_num", "hours_per_week",
                    "capital_gain", "capital_loss"]]
                .corr()["fnlwgt"]
                .drop("fnlwgt")
                .round(4))

print("\nPearson correlation of fnlwgt with each personal attribute:")
display(correlations)
print(f"\nLargest absolute correlation: {correlations.abs().max():.4f}")

fnlwgt summary statistics:


count      48,790.00
mean      189,669.00
std       105,617.23
min        12,285.00
25%       117,555.00
50%       178,138.50
75%       237,606.25
max     1,490,400.00
Name: fnlwgt, dtype: float64


Pearson correlation of fnlwgt with each personal attribute:


age              -0.08
education_num    -0.04
hours_per_week   -0.01
capital_gain     -0.00
capital_loss     -0.00
Name: fnlwgt, dtype: float64


Largest absolute correlation: 0.0765


In [15]:
# ---------------------------------------------------------------------------
# Decision: drop fnlwgt from the analytical frame. It is retained in df_raw for
# any later analysis that needs to produce population-weighted estimates.
# ---------------------------------------------------------------------------
df = df.drop(columns=["fnlwgt"])
print(f"Dropped 'fnlwgt'. Columns remaining: {df.shape[1]}")

Dropped 'fnlwgt'. Columns remaining: 13


### A consequence of dropping `fnlwgt`

Removing a column can create duplicate rows that did not exist before, because two records
distinguished only by the dropped column become identical once it is gone. This is checked
explicitly rather than discovered later.

In [16]:
# ---------------------------------------------------------------------------
# Re-check for duplicates now that 'education' and 'fnlwgt' have been removed.
# These are collisions on the remaining attributes, not the exact-record
# duplicates already handled in section 2.2.
# ---------------------------------------------------------------------------
collisions = int(df.duplicated().sum())
print(f"Rows now identical on all remaining columns: {collisions:,} "
      f"({collisions / len(df) * 100:.2f}% of the frame)")

# Group the frame by every remaining attribute to see the shape of these
# collisions: how many distinct profiles exist, and how often they repeat.
profiles = df.groupby(list(df.columns), dropna=False, observed=True).size()
print(f"Distinct attribute profiles       : {len(profiles):,}")
print(f"Profiles occurring more than once : {int((profiles > 1).sum()):,}")
print(f"Largest number of repeats         : {int(profiles.max())}")

print("\nThe three most frequently repeated profiles:")
display(profiles.sort_values(ascending=False).head(3))

Rows now identical on all remaining columns: 6,322 (12.96% of the frame)


Distinct attribute profiles       : 42,468
Profiles occurring more than once : 3,451
Largest number of repeats         : 21

The three most frequently repeated profiles:


age  workclass  education_num  marital_status      occupation    relationship  race   sex   capital_gain  capital_loss  hours_per_week  native_country  income
33   Private    9              Married-civ-spouse  Craft-repair  Husband       White  Male  0             0             40              United-States   <=50K     21
32   Private    9              Married-civ-spouse  Craft-repair  Husband       White  Male  0             0             40              United-States   <=50K     20
35   Private    9              Married-civ-spouse  Craft-repair  Husband       White  Male  0             0             40              United-States   <=50K     20
dtype: int64

**Finding.** Roughly 13 percent of rows now collide with another row, and the most common
profile repeats 21 times. That profile is a 33-year-old married male working 40 hours a week
in a private-sector craft-repair role. This is not a data defect. It is one of the most
common demographic profiles in the 1994 American workforce, and 21 different respondents
matching it is exactly what a representative sample of nearly 50,000 people should contain.

**Decision.** These collisions are retained. The distinction matters:

- The 52 rows removed in section 2.2 were identical on **all fifteen columns including the
  survey weight**. Two independently sampled respondents receiving byte-identical weights
  by chance is implausible, so those rows carry the signature of a duplication artifact.
- The rows identified here are identical only on **observed attributes**. They are different
  people who happen to share a profile, and their repetition is the frequency information
  that makes the sample representative.

Removing them would flatten common profiles to a single occurrence and systematically
distort the distribution that any later model is fitted against. The count is recorded here
so that a reader does not mistake it for an oversight.

## 2.6 Zero inflation and top-coding in the capital variables

`capital_gain` and `capital_loss` are not ordinary continuous variables. Both are
overwhelmingly zero, and `capital_gain` contains a sentinel value at the top of its range.
Treating either as a plain numeric column would mislead every summary statistic computed
from it.

In [17]:
# ---------------------------------------------------------------------------
# Quantify the zero inflation and locate the top-code.
# ---------------------------------------------------------------------------
for col in ["capital_gain", "capital_loss"]:
    values = df[col].to_numpy()
    pct_zero = np.mean(values == 0) * 100
    nonzero = values[values > 0]
    print(f"{col}")
    print(f"   zero-valued        : {pct_zero:.2f}%")
    print(f"   maximum            : {values.max():,}")
    print(f"   mean over all rows : {values.mean():,.2f}")
    print(f"   mean when non-zero : {nonzero.mean():,.2f}")
    print()

# The maximum of capital_gain is a censoring sentinel rather than a real amount.
n_topcoded = int((df["capital_gain"] == 99999).sum())
print(f"Records at exactly 99999 in capital_gain: {n_topcoded:,}")
print("\nMost frequent non-zero capital_gain values:")
display(df.loc[df["capital_gain"] > 0, "capital_gain"].value_counts().head(6))

capital_gain
   zero-valued        : 91.73%
   maximum            : 99,999
   mean over all rows : 1,080.22
   mean when non-zero : 13,061.67

capital_loss
   zero-valued        : 95.32%
   maximum            : 4,356
   mean over all rows : 87.60
   mean when non-zero : 1,872.83

Records at exactly 99999 in capital_gain: 244

Most frequent non-zero capital_gain values:


capital_gain
15024    513
7688     410
7298     364
99999    244
3103     152
5178     146
Name: count, dtype: int64

**Finding.** Roughly 92 percent of `capital_gain` values and 95 percent of `capital_loss`
values are zero, so the unconditional mean of each column describes almost nobody. The
value 99999 appears far more often than neighboring values, which is the signature of a
**top-code**: the Census Bureau censors values above a threshold to protect respondent
confidentiality. Those records represent "at least 99999 dollars", not "exactly 99999
dollars".

**Decision.** The values are preserved rather than removed, and the structure is made
explicit through indicator columns constructed in section 2.9. Deleting the top-coded
records would discard the highest earners, which is precisely the group the hypothesis is
about.

## 2.7 Outlier assessment

Interquartile-range screening is applied to each numeric column. The results are then
judged against domain plausibility rather than acted on mechanically.

In [18]:
# ---------------------------------------------------------------------------
# Tukey's IQR rule implemented directly with NumPy, so the bounds are visible
# rather than hidden inside a library call. Values beyond 1.5 x IQR from the
# nearer quartile are flagged.
# ---------------------------------------------------------------------------
numeric_cols = ["age", "education_num", "hours_per_week", "capital_gain", "capital_loss"]
rows = []

for col in numeric_cols:
    values = df[col].to_numpy()
    q1, q3 = np.percentile(values, [25, 75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = np.sum((values < lower) | (values > upper))
    rows.append({
        "column": col, "min": values.min(), "Q1": q1, "median": np.median(values),
        "Q3": q3, "max": values.max(), "IQR": iqr,
        "lower_bound": lower, "upper_bound": upper,
        "n_flagged": int(flagged), "pct_flagged": round(flagged / len(values) * 100, 2),
    })

iqr_report = pd.DataFrame(rows).set_index("column")
display(iqr_report)

,min,Q1,median,Q3,max,IQR,lower_bound,upper_bound,n_flagged,pct_flagged
column,,,,,,,,,,
age,17,28.00,37.00,48.00,90,20.00,-2.00,78.00,215,0.44
education_num,1,9.00,10.00,12.00,16,3.00,4.50,16.50,1787,3.66
hours_per_week,1,40.00,40.00,45.00,99,5.00,32.50,52.50,13486,27.64
capital_gain,0,0.00,0.00,0.00,99999,0.00,0.00,0.00,4035,8.27
capital_loss,0,0.00,0.00,0.00,4356,0.00,0.00,0.00,2282,4.68


In [19]:
# ---------------------------------------------------------------------------
# The hours_per_week result needs explaining before it is acted on. Inspect the
# concentration of the distribution that produced it.
# ---------------------------------------------------------------------------
print("Most common hours_per_week values:")
display(df["hours_per_week"].value_counts().head(6))

pct_forty = (df["hours_per_week"] == 40).mean() * 100
print(f"Share of the extract working exactly 40 hours: {pct_forty:.2f}%")
print(f"Observed range: {df['hours_per_week'].min()} to {df['hours_per_week'].max()} hours")

Most common hours_per_week values:


hours_per_week
40    22773
50     4242
45     2715
60     2177
35     1934
20     1860
Name: count, dtype: int64

Share of the extract working exactly 40 hours: 46.68%
Observed range: 1 to 99 hours


**Finding.** The IQR rule flags roughly 28 percent of `hours_per_week` as outlying. This is
an artifact of the method, not a data quality problem. Nearly half the extract reports
exactly 40 hours, which compresses the interquartile range to about 5 hours and places the
upper fence near 52. On that basis a standard 55-hour work week is classified as an
outlier, which is not a defensible reading of the labor market.

**Decision.** No records are removed on the basis of IQR screening. The observed ranges are
checked instead against domain plausibility:

- `age` spans 17 to 90, consistent with the documented extraction filter of over 16
- `hours_per_week` spans 1 to 99, all of which are attainable
- `education_num` spans 1 to 16, the full documented ordinal scale
- the capital variables are addressed through the top-code indicator instead

Every value in the extract is therefore plausible, and mechanical outlier deletion would
remove real observations. This is recorded as an explicit decision rather than an omission.

## 2.8 Rare category consolidation

Several categorical columns contain levels with too few observations to support a stable
estimate. These are consolidated so that later encoding does not produce near-empty
indicator columns.

In [20]:
# ---------------------------------------------------------------------------
# Survey the cardinality of every categorical column before consolidating.
# ---------------------------------------------------------------------------
categorical_cols = ["workclass", "marital_status", "occupation",
                    "relationship", "race", "sex", "native_country"]

cardinality = pd.DataFrame({
    "n_levels": [df[c].nunique() for c in categorical_cols],
    "largest_level_pct": [round(df[c].value_counts(normalize=True).iloc[0] * 100, 2)
                          for c in categorical_cols],
}, index=categorical_cols)
display(cardinality)

,n_levels,largest_level_pct
workclass,9,69.40
marital_status,7,45.84
occupation,15,12.64
relationship,6,40.38
race,5,85.50
sex,2,66.85
native_country,42,89.76


In [21]:
# ---------------------------------------------------------------------------
# native_country: 41 levels dominated by a single value. Levels below a 100-row
# threshold are grouped into 'Other', which keeps the records while removing
# levels too sparse to estimate.
# ---------------------------------------------------------------------------
MIN_LEVEL_SIZE = 100

country_counts = df["native_country"].value_counts()
rare_countries = country_counts[country_counts < MIN_LEVEL_SIZE].index

print(f"native_country levels before  : {df['native_country'].nunique()}")
print(f"Levels below {MIN_LEVEL_SIZE} rows      : {len(rare_countries)}")
print(f"Records affected              : {int(country_counts[rare_countries].sum()):,}")

df["native_country"] = pd.Series(
    np.where(df["native_country"].isin(rare_countries), "Other", df["native_country"]),
    index=df.index,
)

print(f"native_country levels after   : {df['native_country'].nunique()}")
display(df["native_country"].value_counts())

native_country levels before  : 42
Levels below 100 rows      : 26
Records affected              : 1,211
native_country levels after   : 17


native_country
United-States         43792
Other                  1211
Mexico                  943
Unknown                 856
Philippines             294
Germany                 206
Puerto-Rico             184
Canada                  182
El-Salvador             155
India                   151
Cuba                    138
England                 127
China                   122
South                   115
Jamaica                 106
Italy                   105
Dominican-Republic      103
Name: count, dtype: int64

In [22]:
# ---------------------------------------------------------------------------
# workclass: 'Without-pay' and 'Never-worked' together cover only a handful of
# records and describe the same underlying state of being outside paid
# employment. They are merged into a single level.
# ---------------------------------------------------------------------------
print("workclass counts before consolidation:")
display(df["workclass"].value_counts())

df["workclass"] = pd.Series(
    np.where(df["workclass"].isin(["Without-pay", "Never-worked"]),
             "No-pay-or-never-worked", df["workclass"]),
    index=df.index,
)

print("workclass counts after consolidation:")
display(df["workclass"].value_counts())

workclass counts before consolidation:


workclass
Private             33860
Self-emp-not-inc     3861
Local-gov            3136
Unknown              2795
State-gov            1981
Self-emp-inc         1694
Federal-gov          1432
Without-pay            21
Never-worked           10
Name: count, dtype: int64

workclass counts after consolidation:


workclass
Private                   33860
Self-emp-not-inc           3861
Local-gov                  3136
Unknown                    2795
State-gov                  1981
Self-emp-inc               1694
Federal-gov                1432
No-pay-or-never-worked       31
Name: count, dtype: int64

### Overlap between `relationship` and `marital_status`

These two columns are checked for redundancy in the same way `education` was. The result is
different, and the difference matters.

In [23]:
# ---------------------------------------------------------------------------
# Cross-tabulate the two columns to measure how much information they share.
# ---------------------------------------------------------------------------
overlap = pd.crosstab(df["relationship"], df["marital_status"])
display(overlap)

# Measure determinism: for each relationship level, what share of its rows fall
# into that level's single most common marital status?
concentration = overlap.max(axis=1) / overlap.sum(axis=1)
print("Share of each relationship level falling in its most common marital status:")
display((concentration * 100).round(2))

marital_status,Divorced,Married-AF-spouse,Married-civ-spouse,Married-spouse-absent,Never-married,Separated,Widowed
relationship,,,,,,,
Husband,0,12,19691,0,0,0,0
Not-in-family,3626,0,23,329,7091,637,851
Other-relative,181,1,201,54,920,79,70
Own-child,455,1,143,61,6738,146,25
Unmarried,2368,0,0,183,1333,668,572
Wife,0,23,2308,0,0,0,0


Share of each relationship level falling in its most common marital status:


relationship
Husband          99.94
Not-in-family    56.47
Other-relative   61.09
Own-child        89.02
Unmarried        46.21
Wife             99.01
dtype: float64

**Finding.** `Husband` and `Wife` are almost perfectly determined by `Married-civ-spouse`,
so the two columns overlap heavily. They are not redundant, however: `Not-in-family`,
`Own-child`, and `Unmarried` each spread across several marital statuses and carry
household-role information that `marital_status` alone does not.

**Decision.** Both columns are retained. The overlap is documented here so that it can be
addressed at the modeling stage, where collinearity actually affects coefficient estimates.
Removing a column during preparation to solve a modeling problem would be premature.

## 2.9 Feature construction

New columns are derived with NumPy so that the structure identified in the audit is
represented explicitly rather than left implicit in the raw values.

In [24]:
# ---------------------------------------------------------------------------
# Feature 1: binary target. The string label is converted to an integer so that
# it can be used directly as a response variable.
# ---------------------------------------------------------------------------
df["income_gt_50k"] = np.where(df["income"] == ">50K", 1, 0).astype("int8")

# ---------------------------------------------------------------------------
# Feature 2: top-code indicator. Flags the censored capital_gain records so that
# a later model can distinguish 'at least 99999' from a measured amount.
# ---------------------------------------------------------------------------
df["capital_gain_topcoded"] = np.where(df["capital_gain"] == 99999, 1, 0).astype("int8")

# ---------------------------------------------------------------------------
# Feature 3 and 4: participation indicators. Because both capital columns are
# more than 90% zero, whether a person has any capital activity at all is a
# cleaner signal than the amount.
# ---------------------------------------------------------------------------
df["has_capital_gain"] = np.where(df["capital_gain"] > 0, 1, 0).astype("int8")
df["has_capital_loss"] = np.where(df["capital_loss"] > 0, 1, 0).astype("int8")

# ---------------------------------------------------------------------------
# Feature 5: net capital position, which collapses two mostly-zero columns into
# one signed measure.
# ---------------------------------------------------------------------------
df["net_capital"] = df["capital_gain"] - df["capital_loss"]

# ---------------------------------------------------------------------------
# Feature 6: log transform of capital_gain. np.log1p computes log(1 + x), which
# is defined at zero and therefore safe on a zero-inflated column. This
# compresses a range spanning five orders of magnitude.
# ---------------------------------------------------------------------------
df["capital_gain_log"] = np.log1p(df["capital_gain"])

print("Skewness of capital_gain before transform:", round(df["capital_gain"].skew(), 3))
print("Skewness of capital_gain after log1p     :", round(df["capital_gain_log"].skew(), 3))

Skewness of capital_gain before transform: 11.888
Skewness of capital_gain after log1p     : 3.111


In [25]:
# ---------------------------------------------------------------------------
# Feature 7: education tier. np.select evaluates an ordered list of conditions
# and returns the value matching the first true one, which collapses the
# 16-level ordinal into four interpretable bands.
# ---------------------------------------------------------------------------
edu = df["education_num"].to_numpy()

education_conditions = [
    edu <= 8,                  # did not complete high school
    edu == 9,                  # high school graduate
    (edu >= 10) & (edu <= 12),  # some college or associate degree
    edu >= 13,                 # bachelor's degree or higher
]
education_labels = ["No-HS-diploma", "HS-graduate", "Some-college", "Bachelors-plus"]

df["education_tier"] = np.select(education_conditions, education_labels, default="Unknown")

# ---------------------------------------------------------------------------
# Feature 8: working-hours band. The same np.select pattern applied to hours,
# using labor-market conventions rather than the IQR fences rejected in 2.7.
# ---------------------------------------------------------------------------
hours = df["hours_per_week"].to_numpy()

hours_conditions = [
    hours < 35,                    # part time
    (hours >= 35) & (hours <= 40),  # standard full time
    (hours > 40) & (hours <= 50),   # extended
    hours > 50,                     # long hours
]
hours_labels = ["Part-time", "Full-time", "Overtime", "Long-hours"]

df["hours_band"] = np.select(hours_conditions, hours_labels, default="Unknown")

# ---------------------------------------------------------------------------
# Feature 9: age band, using np.digitize to assign each age to a decade bucket.
# ---------------------------------------------------------------------------
age_edges = np.array([25, 35, 45, 55, 65])
age_labels = np.array(["17-24", "25-34", "35-44", "45-54", "55-64", "65-plus"])
df["age_band"] = age_labels[np.digitize(df["age"].to_numpy(), age_edges)]

print("Derived categorical features:\n")
for col in ["education_tier", "hours_band", "age_band"]:
    print(f"--- {col} ---")
    print(df[col].value_counts().sort_index())
    print()

Derived categorical features:

--- education_tier ---
education_tier
Bachelors-plus    12097
HS-graduate       15770
No-HS-diploma      6399
Some-college      14524
Name: count, dtype: int64

--- hours_band ---
hours_band
Full-time     26061
Long-hours     5434
Overtime       8909
Part-time      8386
Name: count, dtype: int64

--- age_band ---
age_band
17-24       8409
25-34      12564
35-44      12184
45-54       8765
55-64       4782
65-plus     2086
Name: count, dtype: int64



In [26]:
# ---------------------------------------------------------------------------
# Verify that no derived column fell through its conditions to the 'Unknown'
# default. A non-zero count here would mean the condition list is incomplete.
# ---------------------------------------------------------------------------
for col in ["education_tier", "hours_band"]:
    fell_through = int((df[col] == "Unknown").sum())
    status = "OK" if fell_through == 0 else "REVIEW"
    print(f"{col:<18} rows falling through to default: {fell_through}  [{status}]")

education_tier     rows falling through to default: 0  [OK]
hours_band         rows falling through to default: 0  [OK]


## 2.10 Type optimization

Assigning the correct dtype to each column reduces memory use and signals intent to any
downstream consumer of the prepared file.

In [27]:
# ---------------------------------------------------------------------------
# Convert low-cardinality text columns to the category dtype, which stores each
# level once and holds integer codes in the column itself.
# ---------------------------------------------------------------------------
to_categorical = ["workclass", "marital_status", "occupation", "relationship",
                  "race", "sex", "native_country", "income",
                  "education_tier", "hours_band", "age_band"]

for col in to_categorical:
    df[col] = df[col].astype("category")

# Downcast the integer columns to the smallest type that holds their range.
df["age"] = df["age"].astype("int8")
df["education_num"] = df["education_num"].astype("int8")
df["hours_per_week"] = df["hours_per_week"].astype("int8")
df["capital_gain"] = df["capital_gain"].astype("int32")
df["capital_loss"] = df["capital_loss"].astype("int32")
df["net_capital"] = df["net_capital"].astype("int32")

df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 48790 entries, 0 to 48789
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   age                    48790 non-null  int8    
 1   workclass              48790 non-null  category
 2   education_num          48790 non-null  int8    
 3   marital_status         48790 non-null  category
 4   occupation             48790 non-null  category
 5   relationship           48790 non-null  category
 6   race                   48790 non-null  category
 7   sex                    48790 non-null  category
 8   capital_gain           48790 non-null  int32   
 9   capital_loss           48790 non-null  int32   
 10  hours_per_week         48790 non-null  int8    
 11  native_country         48790 non-null  category
 12  income                 48790 non-null  category
 13  income_gt_50k          48790 non-null  int8    
 14  capital_gain_topcoded  48790 non-null  int8    
 

## 2.11 Final validation

A set of assertions confirms that the prepared frame satisfies every property the cleaning
steps were intended to produce. An assertion that fails would halt the notebook rather than
allowing a silent defect through to the next micro-project.

In [28]:
# ---------------------------------------------------------------------------
# Each check restates one requirement established during the Prepare step.
# ---------------------------------------------------------------------------
checks = {
    "no missing values remain":
        int(df.isna().sum().sum()) == 0,
    "exact-record duplicates removed":
        len(df) == POST_DEDUP_ROWS,
    "no rows lost after de-duplication":
        len(df) == BASELINE["rows"] - n_dupes,
    "target is strictly binary":
        set(df["income_gt_50k"].unique()) == {0, 1},
    "target label and indicator agree":
        bool((df["income"].eq(">50K").astype(int) == df["income_gt_50k"]).all()),
    "redundant columns removed":
        ("education" not in df.columns) and ("fnlwgt" not in df.columns),
    "age within documented filter":
        bool(df["age"].between(17, 90).all()),
    "hours within plausible range":
        bool(df["hours_per_week"].between(1, 99).all()),
    "education ordinal within scale":
        bool(df["education_num"].between(1, 16).all()),
    "no derived category fell through":
        int((df["education_tier"] == "Unknown").sum()) == 0,
}

for description, passed in checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {description}")

assert all(checks.values()), "One or more validation checks failed."
print("\nAll validation checks passed.")

[PASS] no missing values remain
[PASS] exact-record duplicates removed
[PASS] no rows lost after de-duplication
[PASS] target is strictly binary
[PASS] target label and indicator agree
[PASS] redundant columns removed
[PASS] age within documented filter
[PASS] hours within plausible range
[PASS] education ordinal within scale
[PASS] no derived category fell through

All validation checks passed.


In [29]:
# ---------------------------------------------------------------------------
# Before-and-after comparison quantifying the effect of the Prepare step.
# ---------------------------------------------------------------------------
final_state = {
    "rows": len(df),
    "cols": df.shape[1],
    "memory_mb": df.memory_usage(deep=True).sum() / 1024**2,
    "missing_cells": int(df.isna().sum().sum()),
}

# Duplicate counts are deliberately excluded from this table. The raw frame and
# the prepared frame have different schemas, so a duplicate count computed on one
# is not comparable with a count computed on the other. Both are reported below
# the table with the schema each was measured against.
COMPARABLE = ["rows", "cols", "memory_mb", "missing_cells"]
comparison = pd.DataFrame({
    "before": pd.Series({k: BASELINE[k] for k in COMPARABLE}),
    "after": pd.Series(final_state),
})
comparison["change"] = comparison["after"] - comparison["before"]
display(comparison.round(2))

retained = len(df) / BASELINE["rows"] * 100
print(f"Records retained: {len(df):,} of {BASELINE['rows']:,} ({retained:.2f}%)")
print(f"Memory reduction: {(1 - final_state['memory_mb'] / BASELINE['memory_mb']) * 100:.1f}%")

print("\nDuplicate accounting, stated against the schema each was measured on:")
print(f"  removed, exact duplicates on the raw 15-column schema : {n_dupes:,}")
print(f"  retained, profile collisions on the prepared schema   : {int(df.duplicated().sum()):,}")
print("  (retained by the decision recorded in section 2.5)")

,before,after,change
rows,"48,842.00","48,790.00",-52.00
cols,15.00,22.00,7.00
memory_mb,26.36,1.77,-24.59
missing_cells,"6,465.00",0.00,"-6,465.00"


Records retained: 48,790 of 48,842 (99.89%)
Memory reduction: 93.3%

Duplicate accounting, stated against the schema each was measured on:
  removed, exact duplicates on the raw 15-column schema : 52
  retained, profile collisions on the prepared schema   : 6,326
  (retained by the decision recorded in section 2.5)


In [30]:
# ---------------------------------------------------------------------------
# Confirm that the cleaning did not shift the quantity the project explains.
# The target distribution should be materially unchanged from the raw extract.
# ---------------------------------------------------------------------------
raw_target = df_raw["income"].str.strip().str.rstrip(".")
raw_rate = raw_target.eq(">50K").mean() * 100
clean_rate = df["income_gt_50k"].mean() * 100

print(f">50K rate in the raw extract      : {raw_rate:.2f}%")
print(f">50K rate in the prepared dataset : {clean_rate:.2f}%")
print(f"Shift introduced by preparation   : {clean_rate - raw_rate:+.2f} percentage points")

>50K rate in the raw extract      : 23.93%
>50K rate in the prepared dataset : 23.94%
Shift introduced by preparation   : +0.01 percentage points


In [31]:
# ---------------------------------------------------------------------------
# Final structure of the prepared dataset.
# ---------------------------------------------------------------------------
print(f"Prepared dataset: {df.shape[0]:,} rows x {df.shape[1]} columns\n")
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_unique": df.nunique(),
    "example": df.iloc[0],
})
display(schema)

Prepared dataset: 48,790 rows x 22 columns



,dtype,n_unique,example
age,int8,74,39
workclass,category,8,State-gov
education_num,int8,16,13
marital_status,category,7,Never-married
occupation,category,15,Adm-clerical
relationship,category,6,Not-in-family
race,category,5,White
sex,category,2,Male
capital_gain,int32,123,2174
capital_loss,int32,99,0


In [32]:
# ---------------------------------------------------------------------------
# Export the prepared dataset for use in Micro-Projects 2 through 4.
# ---------------------------------------------------------------------------
OUTPUT_PATH = Path("data/adult_income_prepared.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False)

print(f"Written to {OUTPUT_PATH}")
print(f"File size  : {OUTPUT_PATH.stat().st_size / 1024**2:.2f} MB")
print(f"Dimensions : {df.shape[0]:,} rows x {df.shape[1]} columns")

# Reload and verify the round trip, so the exported artifact is known to be valid.
verify = pd.read_csv(OUTPUT_PATH)
print(f"\nRound-trip check: reloaded {verify.shape[0]:,} rows x {verify.shape[1]} columns")
print(f"Shape preserved : {verify.shape == df.shape}")

Written to data/adult_income_prepared.csv
File size  : 6.40 MB
Dimensions : 48,790 rows x 22 columns



Round-trip check: reloaded 48,790 rows x 22 columns
Shape preserved : True


---

# Scope boundary

This notebook terminates at Step 2 of the data science process, as specified for
Micro-Project 1. The following steps are deliberately not performed here.

| Step | Status | Carried forward to |
|---|---|---|
| 3. Analyze | Not performed | Micro-Project 3 |
| 4. Report | Not performed | Micro-Project 2 |
| 5. Act | Not performed | Micro-Project 3 |

The hypothesis stated at the top of this notebook remains untested by design. The
deliverable of Micro-Project 1 is `adult_income_prepared.csv`, a validated dataset capable
of supporting that test.

### Summary of preparation decisions

| # | Issue found | Decision | Rationale |
|---|---|---|---|
| 1 | `?` sentinel in three columns | Converted to NaN, then to an explicit `Unknown` level | Missingness is structural and informative |
| 2 | 52 exact duplicate rows on the raw schema | Removed | Identical survey weights indicate a duplication artifact |
| 2b | Profile collisions appearing after `fnlwgt` was dropped | Retained | Different respondents sharing a common profile; their frequency is real |
| 3 | `education` duplicates `education_num` | Dropped the text column, kept the ordinal | One-to-one mapping verified |
| 4 | `fnlwgt` is a survey weight | Dropped from the analytical frame | Describes the sample design, not the person |
| 5 | `capital_gain` censored at 99999 | Retained, flagged with an indicator | Removal would delete the highest earners |
| 6 | IQR flags 28% of `hours_per_week` | No records removed | Artifact of a compressed IQR, not a data defect |
| 7 | 26 sparse `native_country` levels | Grouped into `Other` | Too few rows to estimate a level effect |
| 8 | Near-empty `workclass` levels | Merged into one level | Same underlying employment state |
| 9 | `relationship` overlaps `marital_status` | Both retained, overlap documented | Collinearity is a modeling concern, not a cleaning one |
| 10 | Class imbalance at 76/24 | Recorded, not corrected | Resampling during preparation would contaminate evaluation |

### Limitations

The extract is drawn from the 1994 Census and does not describe the contemporary labor
market. Nominal dollar thresholds, occupational categories, and workforce composition have
all changed substantially since collection. Any conclusion drawn in a later micro-project
is a statement about the 1994 sample.

The extract also records `race`, `sex`, and `native_country`. These are retained here
because removing them would obscure rather than remove disparity, and because a later
fairness assessment requires them. Their presence is noted so that any model built on this
prepared file is evaluated for disparate performance across those groups, and so that the
distinction between association and cause is not lost when results are reported.